In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
import joblib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
flights = pd.read_parquet('dataset/merged_flights_fe_v2.parquet')
flights['FL_DATE'] = pd.to_datetime(flights['FL_DATE'])

with open('models/feature_list_final.txt') as f:
    FEATURES = f.read().strip().split('\n')

In [ ]:
CAT_FEATURES = ['OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label']
for col in CAT_FEATURES:
    flights[col] = flights[col].astype('category')

train = flights[flights['FL_DATE'] < '2025-01-01']
test  = flights[flights['FL_DATE'] >= '2025-01-01']

X_train = train[FEATURES]
X_test  = test[FEATURES]
y_train = train['ARR_DELAY']
y_test  = test['ARR_DELAY']

CAP_HIGH = 300
CAP_LOW  = -60
y_train_capped = y_train.clip(lower=CAP_LOW, upper=CAP_HIGH)
y_test_capped  = y_test.clip(lower=CAP_LOW, upper=CAP_HIGH)

del flights, train, test
import gc; gc.collect()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train capped range: [{y_train_capped.min():.0f}, {y_train_capped.max():.0f}]")
print("✓ Ready for Optuna")

X_train: (13708670, 61)
X_test:  (4519126, 61)
y_train capped range: [-60, 300]
✓ Ready for Optuna


In [ ]:
import optuna
import lightgbm as lgb
import numpy as np
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

np.random.seed(42)
train_idx = np.random.choice(len(X_train), 500_000, replace=False)
eval_idx  = np.random.choice(len(X_test),  200_000, replace=False)

X_train_sample = X_train.iloc[train_idx]
y_train_sample = y_train_capped.iloc[train_idx]
X_eval_sample  = X_test.iloc[eval_idx]
y_eval_sample  = y_test_capped.iloc[eval_idx]

print(f"Train sample: {X_train_sample.shape}")
print(f"Eval sample:  {X_eval_sample.shape}")

def objective(trial):
    params = {
        'objective': 'huber',
        'metric': 'mae',
        'verbose': -1,
        'random_state': 42,
        'n_jobs': -1,
        'n_estimators': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 50, 400),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 2.0),
        'max_bin': trial.suggest_int('max_bin', 100, 255),
    }

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train_sample, y_train_sample,
        eval_set=[(X_eval_sample, y_eval_sample)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    )

    preds = model.predict(X_eval_sample)
    return mean_absolute_error(y_eval_sample, preds)

def print_callback(study, trial):
    print(f"Trial {trial.number:>3} | MAE: {trial.value:.4f} | Best: {study.best_value:.4f}")

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=30, show_progress_bar=False, callbacks=[print_callback])

print(f"\n{'='*50}")
print(f"Best MAE: {study.best_value:.4f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Train sample: (500000, 61)
Eval sample:  (200000, 61)
Trial   0 | MAE: 19.4866 | Best: 19.4866
Trial   1 | MAE: 21.1468 | Best: 19.4866
Trial   2 | MAE: 18.4375 | Best: 18.4375
Trial   3 | MAE: 18.7873 | Best: 18.4375
Trial   4 | MAE: 20.7659 | Best: 18.4375
Trial   5 | MAE: 17.6085 | Best: 17.6085
Trial   6 | MAE: 19.2475 | Best: 17.6085
Trial   7 | MAE: 18.2764 | Best: 17.6085
Trial   8 | MAE: 17.8084 | Best: 17.6085
Trial   9 | MAE: 20.4756 | Best: 17.6085
Trial  10 | MAE: 17.5748 | Best: 17.5748
Trial  11 | MAE: 17.5703 | Best: 17.5703
Trial  12 | MAE: 17.5922 | Best: 17.5703
Trial  13 | MAE: 17.9350 | Best: 17.5703
Trial  14 | MAE: 17.5845 | Best: 17.5703
Trial  15 | MAE: 17.7989 | Best: 17.5703
Trial  16 | MAE: 18.6429 | Best: 17.5703
Trial  17 | MAE: 17.6785 | Best: 17.5703
Trial  18 | MAE: 18.9068 | Best: 17.5703
Trial  19 | MAE: 18.1512 | Best: 17.5703
Trial  20 | MAE: 17.7942 | Best: 17.5703
Trial  21 | MAE: 17.5752 | Best: 17.5703
Trial  22 | MAE: 17.6638 | Best: 17.5703
Tri